# prep_04_medicaid
## Ohio Dental Clinic — Site Selection Analysis

**Purpose:** Processes ACS 5-Year Estimates table B27010 (Health Insurance by Age). Sums Medicaid enrollment columns across all age groups for all 1,233 Ohio ZCTAs. Medicaid rate is used as a payer mix quality signal in the composite scoring model (15% weight, inverted — lower Medicaid = higher score).

| | |
|---|---|
| **Input** | `B27010_medicaid_raw.csv` (ACS 2020–2024) |
| **Output** | `B27010_medicaid_cleaned.csv` |
| **Records** | 1,233 Ohio ZCTAs |

In [2]:
# Import libraries
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings("ignore")

# PATHS
RAW_DATA_PATH = r"C:\Users\mosun\Downloads\oh_clinic_rw_files"
OUTPUT_PATH = r"../data/cleaned"

# LOAD
print("Loading B27010_medicaid_raw.csv...")
df = pd.read_csv(
    os.path.join(RAW_DATA_PATH, "ACSDT5Y2024.B27010-Data.csv"),
    header=0,       # row 1 = column codes → used as column names
    skiprows=[1],   # row 2 = human labels → skip it
    dtype=str,
    low_memory=False
)
print(f"Raw file: {len(df)} rows | {len(df.columns)} columns")

# Extract 5-digit ZIP from NAME column
df["zip"] = df["NAME"].str.extract(r"(\d{5})")
df = df.dropna(subset=["zip"])
print(f"After ZIP extraction: {len(df)} rows")

def to_num(col):
    """Convert column to numeric, replacing Census suppression codes with 0."""
    s = pd.to_numeric(df[col], errors="coerce")
    s = s.replace(-666666666, np.nan).replace(-999999999, np.nan)
    return s.fillna(0)

# Sum all 7 Medicaid-related columns across all 4 age groups
df["medicaid_total"] = (
    to_num("B27010_007E") +   # Under 19 — Medicaid only
    to_num("B27010_013E") +   # Under 19 — Medicare + Medicaid dual
    to_num("B27010_023E") +   # 19-34    — Medicaid only
    to_num("B27010_029E") +   # 19-34    — Medicare + Medicaid dual
    to_num("B27010_039E") +   # 35-64    — Medicaid only
    to_num("B27010_046E") +   # 35-64    — Medicare + Medicaid dual
    to_num("B27010_062E")     # 65+      — Medicare + Medicaid dual
)

# Report
n_zero = (df["medicaid_total"] == 0).sum()
print(f"ZIPs with zero Medicaid enrollment: {n_zero}")
print(f"Total Ohio Medicaid enrollees: {df['medicaid_total'].sum():,.0f}")
print(f"Average per ZIP: {df['medicaid_total'].mean():,.0f}")

# Keep final columns
med_clean = df[["zip", "medicaid_total"]].copy()

# SAVE 
med_clean.to_csv(os.path.join(OUTPUT_PATH, "B27010_medicaid_cleaned.csv"), index=False)
print(f"\nSaved: B27010_medicaid_cleaned.csv")
print(f"Rows: {len(med_clean)} | Columns: {list(med_clean.columns)}")
print(f"\nSample:")
print(med_clean.head(5).to_string(index=False))

Loading B27010_medicaid_raw.csv...
Raw file: 1233 rows | 135 columns
After ZIP extraction: 1233 rows
ZIPs with zero Medicaid enrollment: 68
Total Ohio Medicaid enrollees: 2,085,387
Average per ZIP: 1,691

Saved: B27010_medicaid_cleaned.csv
Rows: 1233 | Columns: ['zip', 'medicaid_total']

Sample:
  zip  medicaid_total
43001             268
43002              13
43003             576
43004            5487
43005             142
